# **Import MLflow**

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.default.mlflow_tmp
""")

In [0]:
import mlflow
import mlflow.spark

In [0]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

train = spark.table("train_dataset")
test = spark.table("test_dataset")

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

# **Train**

In [0]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=5
)

model = rf.fit(train)
predictions = model.transform(test)
auc = evaluator.evaluate(predictions)

print("AUC:", auc)

In [0]:
import os

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlflow_tmp"

# **Log Model Run in MLflow**

In [0]:
with mlflow.start_run(run_name="RF_50trees_depth5"):

    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 5)

    mlflow.log_metric("AUC", auc)

    mlflow.spark.log_model(model, "rf_model")

In [0]:
rf2 = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=10
)

model2 = rf2.fit(train)
pred2 = model2.transform(test)
auc2 = evaluator.evaluate(pred2)

with mlflow.start_run():
    
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 10)

    mlflow.log_metric("AUC", auc2)

    mlflow.spark.log_model(model2, "rf_model")

print("Second Model AUC:", auc2)

# **Day - 8**

In [0]:
model

In [0]:
best_model = model   # or whichever performed best

# **Score all Users**

In [0]:
full_data = spark.table("silver_user_features")

In [0]:
from pyspark.ml.feature import VectorAssembler

feature_cols = ["total_events","purchases","total_spent","avg_price"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

scoring_df = assembler.transform(full_data)

In [0]:
predictions = best_model.transform(scoring_df)

# **Select Relevant Columns**

In [0]:
final_predictions = predictions.select(
    "user_id",
    "probability",
    "prediction"
)

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

extract_prob = udf(lambda v: float(v[1]), DoubleType())

In [0]:
final_predictions = predictions.select(
    "user_id",
    extract_prob("probability").alias("purchase_probability"),
    "prediction"
)

# **Save to Gold Delta Table**

In [0]:
final_predictions.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_user_predictions")

In [0]:
spark.sql("SELECT * FROM gold_user_predictions").display()

# **Identify Top Predicted Buyers**

In [0]:
spark.sql("""
SELECT *
FROM gold_user_predictions
ORDER BY purchase_probability DESC
LIMIT 20
""").display()